<div style=" background-color: RGB(0,114,200);" >
<h1 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">PROJET 11 DATA ANALYST</h1>
<h2 style="margin: auto; padding: 20px 0; color:#fff; text-align: center">La poule qui chante : traitement des données
</h2>
</div>

<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - CONFIGURATION ET IMPORTS</h2>
</div>

In [205]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from IPython.display import display

print("🐔 PROJET LA POULE QUI CHANTE - ANALYSE DES MARCHÉS D'EXPORTATION")
print("="*70)

🐔 PROJET LA POULE QUI CHANTE - ANALYSE DES MARCHÉS D'EXPORTATION


<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 2 - CHARGEMENT DES DONNÉES</h2>
</div>

In [206]:

# ============================================================================
#  FONCTION DE CHARGEMENT DES DONNEES
# ============================================================================

def charger_donnees():
    """
    Charge tous les fichiers de données nécessaires à l'analyse
    
    Returns:
        dict: Dictionnaire contenant tous les DataFrames chargés
    """
    print("📂 CHARGEMENT DES DONNÉES")
    print("-" * 30)
    
    donnees = {}
    
    try:
        # 1. Données religieuses (Pew Research)
        print("• Chargement Religious_Composition...")
        donnees['religion'] = pd.read_csv('Religious_Composition_2010-2020_percentages.csv')
        print(f"  ✅ {donnees['religion'].shape[0]} lignes, {donnees['religion'].shape[1]} colonnes")
        
        # 2. Disponibilité alimentaire 2017 (SOURCE PRIORITAIRE)
        print("• Chargement DisponibiliteAlimentaire_2017...")
        donnees['alim_2017'] = pd.read_csv('DisponibiliteAlimentaire_2017.csv')
        print(f"  ✅ {donnees['alim_2017'].shape[0]} lignes, {donnees['alim_2017'].shape[1]} colonnes")
        
        # 3. Population 2000-2018 (SOURCE PRIORITAIRE)
        print("• Chargement Population_2000_2018...")
        donnees['population'] = pd.read_csv('Population_2000_2018.csv')
        print(f"  ✅ {donnees['population'].shape[0]} lignes, {donnees['population'].shape[1]} colonnes")
        
        # 4. Production poulet FAO
        print("• Chargement FAOSTAT_data_production_poulet...")
        donnees['production'] = pd.read_csv('FAOSTAT_data_production_poulet.csv')
        print(f"  ✅ {donnees['production'].shape[0]} lignes, {donnees['production'].shape[1]} colonnes")
        
        # 5. Consommation poulet FAO
        print("• Chargement FAOSTAT_data_consommation_poulet...")
        donnees['consommation'] = pd.read_csv('FAOSTAT_data_consommation_poulet_2010_2018.csv')
        print(f"  ✅ {donnees['consommation'].shape[0]} lignes, {donnees['consommation'].shape[1]} colonnes")
        
        # 6. Import/Export poulet FAO
        print("• Chargement FAOSTAT_data_import_export_poulet...")
        donnees['import_export'] = pd.read_csv('FAOSTAT_data_import_export_poulet.csv')
        print(f"  ✅ {donnees['import_export'].shape[0]} lignes, {donnees['import_export'].shape[1]} colonnes")
        
        # 7. Données World Bank
        print("• Chargement WORLDBANK_data_country...")
        donnees['worldbank'] = pd.read_csv('WORLDBANK_data_country_2010_2018.csv')
        print(f"  ✅ {donnees['worldbank'].shape[0]} lignes, {donnees['worldbank'].shape[1]} colonnes")
        
        # 8. Distances depuis la France
        print("• Chargement Distance_france_nom_iso...")
        donnees['distances'] = pd.read_excel('Distance_france_nom_iso.xlsx', sheet_name='dist_cepii')
        print(f"  ✅ {donnees['distances'].shape[0]} lignes, {donnees['distances'].shape[1]} colonnes")
        
    except Exception as e:
        print(f"❌ Erreur lors du chargement : {e}")
        print("💡 Vérifiez que tous les fichiers CSV sont présents dans le répertoire")
        return None
    
    print(f"\n✅ Chargement terminé - {len(donnees)} fichiers chargés")
    return donnees

# Chargement des données
donnees_brutes = charger_donnees()


📂 CHARGEMENT DES DONNÉES
------------------------------
• Chargement Religious_Composition...
  ✅ 416 lignes, 13 colonnes
• Chargement DisponibiliteAlimentaire_2017...
  ✅ 176600 lignes, 14 colonnes
• Chargement Population_2000_2018...
  ✅ 4411 lignes, 15 colonnes
• Chargement FAOSTAT_data_production_poulet...
  ✅ 7468 lignes, 15 colonnes
• Chargement FAOSTAT_data_consommation_poulet...
  ✅ 7149 lignes, 15 colonnes
• Chargement FAOSTAT_data_import_export_poulet...
  ✅ 2969 lignes, 15 colonnes
• Chargement WORLDBANK_data_country...
  ✅ 1069 lignes, 23 colonnes
• Chargement Distance_france_nom_iso...
  ✅ 224 lignes, 6 colonnes

✅ Chargement terminé - 8 fichiers chargés


<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 3 - EXPLORATION ET IDENTIFICATION DES DOUBLONS</h2>
</div>

In [207]:

# ============================================================================
#  FONCTION D'EXPLORATION DES DONNEES
# ============================================================================

def explorer_donnees(donnees_brutes):
    """
    Explore la structure des données et identifie les doublons potentiels
    
    Args:
        donnees (dict): Dictionnaire des DataFrames chargés
    """
    print("\n🔍 EXPLORATION DES DONNÉES")
    print("=" * 40)
    
    for nom, df in donnees_brutes.items():
        print(f"\n📊 Dataset: {nom.upper()}")
        print("-" * 25)
        print(f"Dimensions: {df.shape}")
        print(f"Colonnes: {list(df.columns)}")
        
        # Identifier les colonnes pays et années
        colonnes_pays = [col for col in df.columns if any(mot in col.lower() 
                        for mot in ['country', 'zone', 'pays', 'area'])]
        colonnes_annee = [col for col in df.columns if any(mot in col.lower() 
                         for mot in ['year', 'annee', 'année']) or col.isdigit()]
        
        if colonnes_pays:
            print(f"Colonnes pays détectées: {colonnes_pays}")
            for col_pays in colonnes_pays:
                pays_uniques = df[col_pays].nunique()
                print(f"  • {col_pays}: {pays_uniques} pays uniques")
        
        if colonnes_annee:
            print(f"Colonnes années détectées: {colonnes_annee}")
            # Pour WorldBank qui a les années en colonnes
            if nom == 'worldbank' and any(col.isdigit() for col in colonnes_annee):
                annees = [col for col in colonnes_annee if col.isdigit()]
                print(f"  • Années: {min(annees)} à {max(annees)}")
            # Pour les autres datasets avec colonne Année
            elif colonnes_annee and not all(col.isdigit() for col in colonnes_annee):
                for col_annee in colonnes_annee:
                    if not col_annee.isdigit():
                        annees_min = df[col_annee].min()
                        annees_max = df[col_annee].max()
                        print(f"  • {col_annee}: {annees_min} à {annees_max}")

def identifier_doublons_potentiels(donnees_brutes):
    """
    Identifie les doublons potentiels entre les sources de données
    
    Args:
        donnees (dict): Dictionnaire des DataFrames chargés
    """
    print("\n⚠️ IDENTIFICATION DES DOUBLONS POTENTIELS")
    print("=" * 50)
    
    # Doublons identifiés selon les spécifications
    doublons_identifies = {
        'Population': ['population','alim_2017',  'worldbank'],
        'Production volaille': ['alim_2017', 'production'],
        'Import/Export': ['alim_2017', 'import_export'],
        'Consommation volaille': ['alim_2017', 'consommation']
    }
    
    print("Doublons identifiés et priorités définies:")
    for categorie, sources in doublons_identifies.items():
        print(f"\n📋 {categorie}:")
        for i, source in enumerate(sources, 1):
            priorite = "🥇 PRIORITÉ" if i == 1 else f"🥈 Backup {i-1}"
            print(f"   {priorite}: {source}")
    
    print("\n💡 Stratégie de résolution:")
    print("• DisponibiliteAlimentaire_2017 > autres sources alimentaires")  
    print("• Population_2000_2018 > autres sources population")


<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 4 - NETTOYAGE ET STANDARDISATION</h2>
</div>

In [208]:

# ============================================================================
#  FONCTION DE NETTOYAGE DES DONNEES REGILIEUSES
# ============================================================================

def nettoyer_donnees_religion(df_religion):
   """
   Nettoie les données de composition religieuse
   
   Args:
       df_religion (DataFrame): Données religieuses brutes
   
   Returns:
       DataFrame: Données religieuses nettoyées
   """
   print("\n🧹 NETTOYAGE - Données religieuses")
   print("-" * 35)

   #Différents 'Level' présent de 1 à 3. 3 pour le monde entier, 2 pour des Groupements continentaux (ex: All asia Pacific) ainsi que 1 pour les pays
   # Ici on filtre donc sur Level = 1
   df_clean = df_religion[df_religion['Level'] == 1].copy()
   print(f"• Filtrage Level=1: {len(df_religion)} → {len(df_clean)} lignes")

   
   # Sélectionner les colonnes d'intérêt
   colonnes_utiles = ['Country', 'Year', 'Christians', 'Muslims', 'Religiously_unaffiliated', 'Buddhists', 
                     'Hindus', 'Jews', 'Other_religions']
   df_clean = df_clean[colonnes_utiles].copy()
   
   # Renommer pour cohérence
   df_clean.rename(columns={
       'Country': 'pays',
       'Year': 'annee',
       'Christians': 'pct_chretiens',
       'Muslims': 'pct_musulmans', 
       'Buddhists': 'pct_bouddhistes',
       'Hindus': 'pct_hindous',
       'Jews': 'pct_juifs',
       'Countrycode': 'code_pays'
   }, inplace=True)
   
   print(f"• Colonnes sélectionnées: {list(df_clean.columns)}")
   print(f"• Période: {df_clean['annee'].min()} - {df_clean['annee'].max()}")
   print(f"• Pays uniques: {df_clean['pays'].nunique()}")
   
   return df_clean


In [209]:


# ============================================================================
#  FONCTION DE NETTOYAGE DES DONNEES ALIMENTAIRES
# ============================================================================

def nettoyer_donnees_alimentaires(df_alim):
   """
   Nettoie les données de disponibilité alimentaire 2017
   
   Args:
       df_alim (DataFrame): Données alimentaires brutes
   
   Returns:
       dict: Dictionnaire avec les données par type (production, import, export, consommation)
   """
   print("\n🧹 NETTOYAGE - Disponibilité alimentaire 2017")
   print("-" * 45)
   
   # Filtrer sur Viande de Volailles uniquement
   df_volaille = df_alim[df_alim['Produit'] == 'Viande de Volailles'].copy()
   print(f"• Filtrage 'Viande de Volailles': {len(df_alim)} → {len(df_volaille)} lignes")
   
   # Identifier les éléments disponibles
   elements_disponibles = df_volaille['Élément'].unique()
   print(f"• Éléments disponibles: {list(elements_disponibles)}")
   
   donnees_alim_clean = {}
   
   # Production
   prod_mask = df_volaille['Élément'] == 'Production'
   if prod_mask.any():
       donnees_alim_clean['production'] = df_volaille[prod_mask][['Zone', 'Valeur', 'Unité']].copy()
       donnees_alim_clean['production'].rename(columns={
           'Zone': 'pays',
           'Valeur': 'production_tonnes'
       }, inplace=True)
       donnees_alim_clean['production']['annee'] = 2017
       print(f"• Production: {len(donnees_alim_clean['production'])} pays")
   
   # Importations
   import_mask = df_volaille['Élément'] == 'Importations - Quantité'
   if import_mask.any():
       donnees_alim_clean['importations'] = df_volaille[import_mask][['Zone', 'Valeur', 'Unité']].copy()
       donnees_alim_clean['importations'].rename(columns={
           'Zone': 'pays', 
           'Valeur': 'importations_tonnes'
       }, inplace=True)
       donnees_alim_clean['importations']['annee'] = 2017
       print(f"• Importations: {len(donnees_alim_clean['importations'])} pays")
   
   # Exportations
   export_mask = df_volaille['Élément'] == 'Exportations - Quantité'
   if export_mask.any():
       donnees_alim_clean['exportations'] = df_volaille[export_mask][['Zone', 'Valeur', 'Unité']].copy()
       donnees_alim_clean['exportations'].rename(columns={
           'Zone': 'pays',
           'Valeur': 'exportations_tonnes'
       }, inplace=True)
       donnees_alim_clean['exportations']['annee'] = 2017
       print(f"• Exportations: {len(donnees_alim_clean['exportations'])} pays")
   
   # Consommation (Nourriture)
   consom_mask = df_volaille['Élément'] == 'Nourriture'
   if consom_mask.any():
       donnees_alim_clean['consommation'] = df_volaille[consom_mask][['Zone', 'Valeur', 'Unité']].copy()
       donnees_alim_clean['consommation'].rename(columns={
           'Zone': 'pays',
           'Valeur': 'consommation_tonnes'
       }, inplace=True)
       donnees_alim_clean['consommation']['annee'] = 2017
       print(f"• Consommation: {len(donnees_alim_clean['consommation'])} pays")
   
   return donnees_alim_clean


In [210]:


# ============================================================================
#  FONCTION DE NETTOYAGE DES DONNEES POPULATION
# ============================================================================

def nettoyer_donnees_population(df_pop):
   """
   Nettoie les données de population et extrait la plus récente par pays
   
   Args:
       df_pop (DataFrame): Données de population brutes
   
   Returns:
       DataFrame: Données de population avec année la plus récente par pays
   """
   print("\n🧹 NETTOYAGE - Population")
   print("-" * 25)
   
   # Nettoyer les colonnes
   df_clean = df_pop.copy()
   df_clean.rename(columns={
       'Zone': 'pays',
       'Année': 'annee', 
       'Valeur': 'population_milliers'
   }, inplace=True)
   
   # Supprimer les valeurs nulles
   df_clean = df_clean.dropna(subset=['population_milliers'])
   print(f"• Suppression valeurs nulles: {len(df_pop)} → {len(df_clean)} lignes")
   
   # Extraire l'année la plus récente par pays
   df_recent = df_clean.loc[df_clean.groupby('pays')['annee'].idxmax()].copy()
   print(f"• Extraction année la plus récente: {len(df_clean)} → {len(df_recent)} lignes")
   
   # Statistiques
   print(f"• Période des données: {df_clean['annee'].min()} - {df_clean['annee'].max()}")
   print(f"• Pays uniques: {df_recent['pays'].nunique()}")
   print(f"• Années les plus récentes utilisées:")
   
   repartition_annees = df_recent['annee'].value_counts().sort_index()
   for annee, count in repartition_annees.items():
       print(f"    {annee}: {count} pays")
   
   return df_recent[['pays', 'annee', 'population_milliers']]


In [211]:


# ============================================================================
#  FONCTION DE NETTOYAGE DES DONNEES DE LA BANQUE MONDIALE (WORLD BANK)
# ============================================================================

def transformer_donnees_worldbank(df_wb):
    """
    Transforme les données World Bank du format wide vers long et filtre les séries pertinentes
    
    Args:
        df_wb (DataFrame): Données World Bank en format wide
    
    Returns:
        dict: Dictionnaire avec les données par série
    """
    print("\n🧹 NETTOYAGE - Données World Bank")
    print("-" * 35)
    
    # Séries d'intérêt identifiées
    series_pertinentes = {
        'GDP (current US$)': 'pib_total_usd',
        'GDP growth (annual %)': 'croissance_pib_pct',
        'Urban population (% of total population)': 'urbanisation_pct',
        'Political Stability and Absence of Violence/Terrorism: Estimate': 'stabilite_politique'
    }
    
    print(f"• Séries recherchées: {list(series_pertinentes.keys())}")
    
    # Filtrer sur les séries pertinentes (déjà ciblé en amont mais utile pour un import futur plus exhaustif)
    df_filtre = df_wb[df_wb['Series Name'].isin(series_pertinentes.keys())].copy()
    print(f"• Filtrage séries: {len(df_wb)} → {len(df_filtre)} lignes")
    
    # Debug pour voir les colonnes
    print(f"• Toutes les colonnes: {list(df_filtre.columns)}")
    
    # Identifier les colonnes années (format '2000 [YR2000]')
    colonnes_annees = [col for col in df_filtre.columns 
                      if isinstance(col, str) and '[YR' in col and ']' in col]
    
    # Si pas trouvé avec ce format, essayer d'autres formats
    if not colonnes_annees:
        colonnes_annees = [col for col in df_filtre.columns if str(col).isdigit() and len(str(col)) == 4]
    
    if not colonnes_annees:
        colonnes_annees = [col for col in df_filtre.columns if '20' in str(col) and len(str(col)) >= 4]
    
    print(f"• Colonnes années trouvées: {len(colonnes_annees)} exemples: {colonnes_annees[:3]}")
    
    if not colonnes_annees:
        print("❌ Aucune colonne année trouvée")
        return {}
    
    donnees_wb_clean = {}
    
    for serie_originale, serie_clean in series_pertinentes.items():
        # Filtrer sur la série
        df_serie = df_filtre[df_filtre['Series Name'] == serie_originale].copy()
        
        if len(df_serie) == 0:
            print(f"⚠️  Série '{serie_originale}' non trouvée")
            continue
            
        # Transformer de wide à long
        df_long = df_serie.melt(
            id_vars=['Country Name', 'Country Code'],
            value_vars=colonnes_annees,
            var_name='annee_brute',
            value_name=serie_clean
        )
        
        # Extraire l'année du format '2000 [YR2000]' ou autres formats
        def extraire_annee(col_name):
            if '[YR' in str(col_name) and ']' in str(col_name):
                # Format '2000 [YR2000]' -> extraire la première partie
                return int(str(col_name).split(' [')[0])
            elif str(col_name).isdigit():
                # Format direct '2000'
                return int(col_name)
            else:
                # Essayer d'extraire les 4 premiers chiffres
                import re
                match = re.search(r'(\d{4})', str(col_name))
                if match:
                    return int(match.group(1))
                else:
                    return None
        
        # Appliquer l'extraction d'année
        df_long['annee'] = df_long['annee_brute'].apply(extraire_annee)
        
        # Supprimer les lignes où l'année n'a pas pu être extraite
        df_long = df_long.dropna(subset=['annee'])
        df_long['annee'] = df_long['annee'].astype(int)

        #convertir données worldbank en données numériques
        df_long[serie_clean] = pd.to_numeric(df_long[serie_clean], errors='coerce')


        
        # Supprimer la colonne temporaire et les valeurs nulles
        df_long = df_long.drop('annee_brute', axis=1)
        df_long = df_long.dropna(subset=[serie_clean])
        
        # Extraire année la plus récente par pays
        if len(df_long) > 0:
            df_recent = df_long.loc[df_long.groupby('Country Name')['annee'].idxmax()].copy()
            df_recent.rename(columns={'Country Name': 'pays', 'Country Code': 'code_pays'}, inplace=True)
            
            donnees_wb_clean[serie_clean] = df_recent[['pays', 'code_pays', 'annee', serie_clean]]
            print(f"• {serie_clean}: {len(df_recent)} pays (années {df_recent['annee'].min()}-{df_recent['annee'].max()})")
        
    return donnees_wb_clean


In [212]:
# ============================================================================
#  FONCTION D'EXTRACTION DES DONNEES RELIGIEUSES RECENTES
# ============================================================================

def extraire_donnees_recentes_religion(df_religion):
    """
    Extrait les données religieuses les plus récentes par pays
    
    Args:
        df_religion (DataFrame): Données religieuses nettoyées
        
    Returns:
        DataFrame: Données religieuses avec année la plus récente par pays
    """
    print("\n📅 EXTRACTION - Données religieuses les plus récentes")
    print("-" * 52)
    
    # Supprimer les lignes avec des valeurs nulles dans les colonnes principales
    cols_religion = ['pct_chretiens', 'pct_musulmans', 'Religiously_unaffiliated', 'pct_bouddhistes', 'pct_hindous', 'pct_juifs', 'Other_religions']
    df_clean = df_religion.dropna(subset=cols_religion, how='all')
    
    # Extraire l'année la plus récente par pays
    df_recent = df_clean.loc[df_clean.groupby('pays')['annee'].idxmax()].copy()
    
    print(f"• Pays avec données religieuses: {len(df_recent)}")
    print(f"• Période: {df_recent['annee'].min()} - {df_recent['annee'].max()}")
    
    # Calculer la religion majoritaire
    cols_pct = ['pct_chretiens', 'pct_musulmans', 'Religiously_unaffiliated', 'pct_bouddhistes', 'pct_hindous', 'pct_juifs', 'Other_religions']
    df_recent['religion_majoritaire'] = df_recent[cols_pct].idxmax(axis=1)
    df_recent['pct_religion_majoritaire'] = df_recent[cols_pct].max(axis=1)

    # Créer la variable Pop_traitement_viande(%) :indiquant le % de la population concernée par un traitement particulier de la viande
    df_recent['Pop_traitement_viande(%)'] = (
        df_recent['pct_musulmans'].fillna(0) + 
        df_recent['pct_juifs'].fillna(0) + 
        df_recent['pct_bouddhistes'].fillna(0)
    )
    
    # Nettoyer les noms de religions
    mapping_religions = {
        'pct_chretiens': 'Chrétiens',
        'pct_musulmans': 'Musulmans', 
        'pct_bouddhistes': 'Bouddhistes',
        'pct_hindous': 'Hindous',
        'pct_juifs': 'Juifs',
        'Religiously_unaffiliated': 'Non affilié',
        'Other_religions': 'autres'
    }
    df_recent['religion_majoritaire'] = df_recent['religion_majoritaire'].map(mapping_religions)
    
    print(f"• Répartition religions majoritaires:")
    repartition = df_recent['religion_majoritaire'].value_counts()
    for religion, count in repartition.items():
        print(f"    {religion}: {count} pays")
    
    return df_recent



In [213]:

# ============================================================================
# APPLICATION DE L'EXTRACTION
# ============================================================================


print("\n" + "="*70) 
print("📅 EXTRACTION DES DONNÉES LES PLUS RÉCENTES")
print("="*70)

#Création d'un dictionnaire de données
donnees_propres = {}

if 'religion' in donnees_propres:
    donnees_propres['religion_recente'] = extraire_donnees_recentes_religion(donnees_propres['religion'])


📅 EXTRACTION DES DONNÉES LES PLUS RÉCENTES


In [214]:


#  ==========================================================================================
#  FONCTION DE NETTOYAGE DES DONNEES DE DISTANCE PAR RAPPORT A LA FRANCE (CAPITALE A CAPITAL)
#  ==========================================================================================

def nettoyer_donnees_distances(df_dist):
   """
   Nettoie les données de distances depuis la France
   
   Args:
       df_dist (DataFrame): Données de distances brutes
   
   Returns:
       DataFrame: Données de distances nettoyées
   """
   print("\n🧹 NETTOYAGE - Distances depuis la France")
   print("-" * 40)
   
   # Sélectionner les colonnes utiles (ignorer c_num comme demandé)
   colonnes_utiles = ['iso3_d', 'iso2', 'dist_cap', 'nom_pays', 'country_name']
   df_clean = df_dist[colonnes_utiles].copy()
   
   # Renommer pour cohérence
   df_clean.rename(columns={
       'nom_pays': 'pays_fr',
       'country_name': 'pays_en', 
       'dist_cap': 'distance_france_km'
   }, inplace=True)
   
   df_clean['distance_france_km'] = pd.to_numeric(df_clean['distance_france_km'], errors='coerce')

   # Supprimer les valeurs nulles
   df_clean = df_clean.dropna(subset=['distance_france_km'])
   
   print(f"• Colonnes conservées: {list(df_clean.columns)}")
   print(f"• Pays avec distance: {len(df_clean)}")
   print(f"• Distance min: {df_clean['distance_france_km'].min():.0f} km")
   print(f"• Distance max: {df_clean['distance_france_km'].max():.0f} km")
   
   return df_clean

In [215]:


# ============================================================================
#  APPLICATION DU NETTOYAGE DES DONNEES 
# ============================================================================


#print("\n" + "="*70)
#print("🧹 PHASE DE NETTOYAGE DES DONNÉES")
#print("="*70)



#if donnees_brutes:
   # Nettoyage religion
#   donnees_propres['religion'] = nettoyer_donnees_religion(donnees_brutes['religion'])
   
   # Nettoyage alimentaire (source prioritaire)
#   donnees_propres['alimentaire'] = nettoyer_donnees_alimentaires(donnees_brutes['alim_2017'])
   
   # Nettoyage population (source prioritaire) 
#   donnees_propres['population'] = nettoyer_donnees_population(donnees_brutes['population'])
   
   # Nettoyage World Bank
#   donnees_propres['worldbank'] = transformer_donnees_worldbank(donnees_brutes['worldbank'])
   
   # Nettoyage distances
#   donnees_propres['distances'] = nettoyer_donnees_distances(donnees_brutes['distances'])

In [216]:
#print("Clés après nettoyage:")
#print(list(donnees_propres.keys()))

In [217]:

# ============================================================================
#  AFFICHAGE DES DONNEES NETTOYEES
# ============================================================================


#print("\n" + "="*70)
#print("📊 AFFICHAGE DES DONNÉES NETTOYÉES")
# 1. DONNÉES RELIGIEUSES
#print("🛐 === DONNÉES RELIGIEUSES ===")
#display(donnees_propres['religion'])

# 2. DONNÉES ALIMENTAIRES
#print("\n🍗 === DONNÉES ALIMENTAIRES ===")
#elements = ['production', 'importations', 'exportations', 'consommation']

#for element in elements:
#    print(f"\n=== {element.upper()} ===")
#    display(donnees_propres['alimentaire'][element])

# 3. DONNÉES DE POPULATION
#print("\n👥 === DONNÉES DE POPULATION ===")
#display(donnees_propres['population'])

# 4. DONNÉES WORLD BANK
#print("\n💰 === DONNÉES WORLD BANK ===")
#series_wb = ['pib_total_usd', 'croissance_pib_pct', 'urbanisation_pct', 'stabilite_politique']

#for serie in series_wb:
#    print(f"\n=== {serie.upper().replace('_', ' ')} ===")
#    display(donnees_propres['worldbank'][serie])

# 5. DONNÉES DE DISTANCES
#print("\n🌍 === DISTANCES DEPUIS LA FRANCE ===")
#display(donnees_propres['distances'])

In [ ]:
# Application des fonctions d'exploration et nettoyage
if donnees_brutes:
    # 1. Explorer la structure générale
    explorer_donnees(donnees_brutes)
    
    # 2. Identifier les doublons potentiels
    identifier_doublons_potentiels(donnees_brutes)
    
    # 3. Nettoyer chaque dataset individuellement
    print("\n🧹 PHASE DE NETTOYAGE")
    print("=" * 40)
    
    # Nettoyage des données religieuses
    df_religion_clean = nettoyer_donnees_religion(donnees_brutes['religion'])
    
    # df_population_clean = nettoyer_donnees_population(donnees_brutes['population'])
    # df_alim_clean = nettoyer_donnees_alimentaires(donnees_brutes['alim_2017'])
    # etc.
    
    print("\n✅ Nettoyage terminé")
else:
    print("❌ Impossible de continuer - données non chargées")


🔍 EXPLORATION DES DONNÉES

📊 Dataset: RELIGION
-------------------------
Dimensions: (416, 13)
Colonnes: ['Region', 'Country', 'Year', 'Population', 'Christians', 'Muslims', 'Religiously_unaffiliated', 'Buddhists', 'Hindus', 'Jews', 'Other_religions', 'Level', 'Countrycode']
Colonnes pays détectées: ['Country', 'Countrycode']
  • Country: 208 pays uniques
  • Countrycode: 208 pays uniques
Colonnes années détectées: ['Year']
  • Year: 2010 à 2020

📊 Dataset: ALIM_2017
-------------------------
Dimensions: (176600, 14)
Colonnes: ['Code Domaine', 'Domaine', 'Code zone', 'Zone', 'Code Élément', 'Élément', 'Code Produit', 'Produit', 'Code année', 'Année', 'Unité', 'Valeur', 'Symbole', 'Description du Symbole']
Colonnes pays détectées: ['Code zone', 'Zone']
  • Code zone: 174 pays uniques
  • Zone: 174 pays uniques
Colonnes années détectées: ['Code année', 'Année']
  • Code année: 2017 à 2017
  • Année: 2017 à 2017

📊 Dataset: POPULATION
-------------------------
Dimensions: (4411, 15)
Colo

<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 5 - STANDARDISATION AVEC FICHIER DE RÉFÉRENCE</h2>
</div>

In [219]:

# ============================================================================
#  FONCTION DE NETTOYAGE DES DOUBLONS DU FICHIER DE REFERENCE PAYS
# ============================================================================

def nettoyer_doublons_ref_pays():
   """
   Vérifie et supprime les doublons dans le fichier de référence des pays
   
   Returns:
       pd.DataFrame: DataFrame nettoyé sans doublons
   """
   print("🔍 NETTOYAGE DES DOUBLONS - FICHIER REF_PAYS")
   print("=" * 50)
   
   try:
       # Charger le fichier original
       df_ref = pd.read_excel('ref_pays.xlsx')
       print(f"📊 Nombre total de lignes: {len(df_ref)}")
       
       # Vérifier les doublons sur la colonne 'country' uniquement
       doublons_country = df_ref[df_ref.duplicated(subset=['country'], keep=False)]
       if len(doublons_country) > 0:
           print(f"\n⚠️ DOUBLONS DÉTECTÉS sur 'country': {len(doublons_country)} lignes")
           print(doublons_country[['country', 'pays']].sort_values('country'))
       
       # Supprimer les doublons (garder le premier)
       df_ref_clean = df_ref.drop_duplicates(subset=['country'], keep='first')
       
       doublons_supprimes = len(df_ref) - len(df_ref_clean)
       print(f"\n✅ NETTOYAGE TERMINÉ:")
       print(f"   Lignes avant: {len(df_ref)}")
       print(f"   Lignes après: {len(df_ref_clean)}")
       print(f"   Doublons supprimés: {doublons_supprimes}")
       
       # Sauvegarder le fichier nettoyé
       df_ref_clean.to_excel('ref_pays_clean.xlsx', index=False)
       print(f"✅ Fichier nettoyé sauvegardé: ref_pays_clean.xlsx")
       
       return df_ref_clean
       
   except Exception as e:
       print(f"❌ Erreur lors du nettoyage: {e}")
       return pd.DataFrame()

In [220]:

# ============================================================================
#  FONCTION DE CHARGEMENT DES DONNEES DE REFERENCES PAYS AVEC MAPPINGS
# ============================================================================

def charger_reference_pays():
    """
    Charge le fichier de référence des pays et crée les dictionnaires de mapping
    
    Returns:
        tuple: (mapping_fr_en, mapping_en_en, df_reference)
    """
    print("\n🗺️ CHARGEMENT FICHIER DE RÉFÉRENCE PAYS")
    print("-" * 45)
    
    try:
        # Charger le fichier de référence
        df_ref = pd.read_excel('ref_pays_clean.xlsx')  # ou .csv selon votre format
        
        print(f"✅ Fichier référence chargé: {len(df_ref)} pays")
        print(f"   Colonnes: {list(df_ref.columns)}")
        
        # Nettoyer les données de référence
        df_ref = df_ref.dropna(subset=['country', 'pays'])
        df_ref['country'] = df_ref['country'].str.strip()
        df_ref['pays'] = df_ref['pays'].str.strip()
        
        # Créer dictionnaires de mapping
        mapping_fr_en = dict(zip(df_ref['pays'], df_ref['country']))
        #mapping_en_en = dict(zip(df_ref['country'], df_ref['country']))  # Normalisation anglais
        

        mappings_speciaux = {
   # === MAPPINGS ===
   'Iran (Islamic Republic of)': 'Iran',
   'Venezuela (Bolivarian Republic of)': 'Venezuela',
   'Bolivia (Plurinational State of)': 'Bolivia',
   'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
   'Czech Republic': 'Czechia',
   'Republic of Moldova': 'Moldova',
   
   # === VARIANTES RELIGION (ANGLAIS) ===
   'Bosnia-Herzegovina': 'Bosnia and Herzegovina',
   'Brunei': 'Brunei Darussalam',
   'Channel Islands': 'United Kingdom',
   'East Timor': 'Timor-Leste',
   'Ivory Coast': 'Cote d\'Ivoire',
   'Palestine': 'Palestine',
   'Serbia-Montenegro': 'Serbia',
   'Vatican': 'Vatican City',
   'Macedonia': 'North Macedonia',
   'Myanmar (Burma)': 'Myanmar',
   'Congo Republic': 'Congo',
   'Congo Democratic Republic': 'Democratic Republic of Congo',
   'Tanzania': 'United Republic of Tanzania',
   
   # === VARIANTES ALIMENTAIRE (FRANÇAIS → ANGLAIS) ===
   'Émirats arabes unis': 'United Arab Emirates',
   'Équateur': 'Ecuador',
   'Éthiopie': 'Ethiopia',
   'République islamique d\'Iran': 'Iran',
   'Venezuela (République bolivarienne du)': 'Venezuela',
   'Bolivie (État plurinational de)': 'Bolivia',
   'République-Unie de Tanzanie': 'Tanzania',
   'République démocratique du Congo': 'Democratic Republic of Congo',
   'République du Congo': 'Congo',
   'République centrafricaine': 'Central African Republic',
   'République dominicaine': 'Dominican Republic',
   'Royaume-Uni de Grande-Bretagne et d\'Irlande du Nord': 'United Kingdom',
   'République arabe syrienne': 'Syria',
   'République démocratique populaire lao': 'Laos',
   'ex-République yougoslave de Macédoine': 'North Macedonia',
   
   # === VARIANTES FAO SPÉCIALES ===
   'Cabo Verde': 'Cape Verde',
   'Eswatini': 'Swaziland',
   'Türkiye': 'Turkey',
   'Myanmar': 'Myanmar',
   'Lao People\'s Democratic Republic': 'Laos',
   'Syrian Arab Republic': 'Syria',
   'Republic of the Congo': 'Congo',
   'Democratic Republic of the Congo': 'Democratic Republic of Congo',
   'Central African Republic': 'Central African Republic',
   'United Republic of Tanzania': 'Tanzania',
   
   # === TERRITOIRES ET CAS SPÉCIAUX ===
   'Hong Kong SAR, China': 'Hong Kong',
   'Macao SAR, China': 'Macao',
   'Taiwan Province of China': 'Taiwan',
   'China, Taiwan Province of': 'Taiwan',
   'Palestinian Territory': 'Palestine',
   'West Bank and Gaza': 'Palestine',
   'Occupied Palestinian Territory': 'Palestine',

   # Variantes Chine
   'Chine, continentale': 'China',
   'Chine, Taiwan Province de': 'Taiwan',
   'Chine - RAS de Hong-Kong': 'Hong Kong',

   # Variantes pays
   'Côte d\'Ivoire': 'Cote d\'Ivoire',
   'Iran (République islamique d\')': 'Iran', 
   'Macédoine du Nord': 'North Macedonia',
   'Monténégro': 'Montenegro',
   'République de Moldova': 'Moldova',
   'Tanzania': 'Tanzania',
   'VietNam': 'Vietnam',
   'Dominica': 'Dominica',

   'Bélarus': 'Belarus', 
   'Tchéquie': 'Czech Republic',
   'Czechia': 'Czech Republic',
   'Tchèque (la République)': 'Czech Republic',
   'Égypte': 'Egypt',
   'Saint-Kitts-et-Nevis': 'Saint Kitts and Nevis',
   'Chine - RAS de Macao': 'Macao',
   'Kazakhstan': 'Kazakstan',
   'Timor-Leste': 'Timor-Leste',

   # Variantes World Bank corrigées
   'Kirghizistan': 'Kyrgyzstan',
   'Serbie': 'Serbia',
   
   # === PAYS IMPORTANTS RÉCUPÉRÉS ===
   'United States': 'United States of America',  # Version courte → référence
   'États-Unis d\'Amérique': 'United States of America',  # Français → référence
   'Fédération de Russie': 'Russian Federation',  # Français → référence
   'Republic of Korea': 'Korea',  # Version longue → référence
   'République de Corée': 'Korea',  # Français → référence
   'Turquie': 'Turkey',  # Français → référence
}
        
        # Fusionner les mappings
        mapping_complet = {**mapping_fr_en, **mappings_speciaux}

        
        print(f"✅ Dictionnaire créé: {len(mapping_complet)} correspondances")
        print(f"   Français → Anglais: {len(mapping_fr_en)}")
        #print(f"   Anglais → Anglais: {len(mapping_en_en)}")
        print(f"   Mappings spéciaux: {len(mappings_speciaux)}")
        
        return mapping_complet, df_ref
        
    except Exception as e:
        print(f"❌ Erreur chargement référence: {e}")
        return {}, pd.DataFrame()

In [221]:


# ============================================================================
#  FONCTION DE STANDARDISATION DES REFERENCES DE PAYS
# ============================================================================

def standardize_countries_reference(df, country_col, mapping_dict):
    """
    Standardise les noms de pays avec le fichier de référence
    
    Args:
        df (DataFrame): DataFrame à standardiser
        country_col (str): Nom de la colonne pays
        mapping_dict (dict): Dictionnaire de correspondances
        
    Returns:
        DataFrame: DataFrame avec noms standardisés
    """
    if country_col not in df.columns:
        return df
    
    print(f"   🔄 Standardisation colonne '{country_col}'...")
    
    # Nettoyer les noms avant mapping
    df[country_col] = df[country_col].str.strip()
    
    # Compter les noms avant standardisation
    noms_avant = df[country_col].nunique()
    
    # Appliquer le mapping (garde le nom original si pas trouvé)
    df[country_col] = df[country_col].map(mapping_dict).fillna(df[country_col])
    
    # Compter après standardisation  
    noms_apres = df[country_col].nunique()
    correspondances = df[country_col].isin(mapping_dict.values()).sum()
    
    print(f"   ✅ {noms_avant} → {noms_apres} noms uniques")
    print(f"   ✅ {correspondances}/{len(df)} lignes standardisées")
    
    return df

In [222]:



# ============================================================================
#  FONCTION DE STANDARDISATION COMPLETE
# ============================================================================

def appliquer_standardisation_complete(donnees_brutes):
    """
    Applique la standardisation à toutes les sources de données
    
    Args:
        donnees_brutes (dict): Dictionnaire des données brutes
        
    Returns:
        dict: Données avec noms de pays standardisés
    """
    print("\n🔧 STANDARDISATION COMPLÈTE AVEC RÉFÉRENCE")
    print("=" * 50)
    
    # Charger le fichier de référence
    mapping_complet, df_ref = charger_reference_pays()
    
    if not mapping_complet:
        print("❌ Impossible de continuer sans fichier de référence")
        return donnees_brutes
    
    donnees_standardisees = {}
    
    # Configuration des colonnes pays par source
    config_colonnes = {
        'religion': 'Country',
        'alim_2017': 'Zone',
        'population': 'Zone', 
        'production': 'Zone',
        'consommation': 'Zone',
        'trade': 'Zone',
        'worldbank': 'Country Name',
        'distances': 'nom_pays'
    }
    
    # Appliquer la standardisation
    for source_key, df in donnees_brutes.items():
        print(f"\n📍 Source: {source_key}")
        
        if source_key in config_colonnes:
            country_col = config_colonnes[source_key]
            if source_key == 'religion':
                exclusions_regionales = [
                'All Asia-Pacific', 'All Europe', 'All Latin America-Caribbean',
                'All Middle East-North Africa', 'All North America', 'All Sub-Saharan Africa',
                'All World'
            ]
            df = df[~df[country_col].isin(exclusions_regionales)].copy()
            print(f"   🚫 Exclusions régionales: {len(exclusions_regionales)} types supprimés")
            df_standardise = standardize_countries_reference(df.copy(), country_col, mapping_complet)
            donnees_standardisees[source_key] = df_standardise
        else:
            print(f"   ⚠️  Configuration manquante, données inchangées")
            donnees_standardisees[source_key] = df
    
    return donnees_standardisees

# =============================================================================
# FONCTION DE DEBUG AVANCÉ
# =============================================================================

def debug_correspondances_avec_reference(donnees_propres, df_reference):
    """
    Debug avancé des correspondances avec le fichier de référence
    
    Args:
        donnees_propres (dict): Données nettoyées
        df_reference (DataFrame): Fichier de référence pays
    """
    print("\n🔍 DEBUG CORRESPONDANCES AVEC RÉFÉRENCE")
    print("-" * 45)
    
    # Pays de référence (anglais)
    pays_reference = set(df_reference['country'].dropna().str.strip())
    
    # Pays alimentaire (base)
    if 'alimentaire' in donnees_propres and 'production' in donnees_propres['alimentaire']:
        pays_alim = set(donnees_propres['alimentaire']['production']['pays'].unique())
        print(f"📊 Pays alimentaires: {len(pays_alim)}")
        print(f"📊 Pays référence: {len(pays_reference)}")
        
        # Correspondances
        correspondances_alim = pays_alim.intersection(pays_reference)
        manquants_alim = pays_alim - pays_reference
        
        print(f"✅ Correspondances: {len(correspondances_alim)}/{len(pays_alim)} ({len(correspondances_alim)/len(pays_alim)*100:.1f}%)")
        
        if manquants_alim:
            print(f"\n❌ Pays alimentaires non trouvés dans référence ({len(manquants_alim)}):")
            for pays in sorted(list(manquants_alim)[:10]):
                print(f"   • {pays}")
        
        # Autres sources
        sources_a_tester = {
            'religion_recente': 'pays',
            'population': 'pays'
        }
        
        if 'worldbank' in donnees_propres:
            # World Bank : collecter de toutes les séries
            pays_wb = set()
            for serie, df in donnees_propres['worldbank'].items():
                pays_wb.update(df['pays'].unique())
            sources_a_tester['worldbank'] = pays_wb
        
        for source, col_ou_set in sources_a_tester.items():
            if source in donnees_propres:
                if isinstance(col_ou_set, str):
                    pays_source = set(donnees_propres[source][col_ou_set].unique())
                else:
                    pays_source = col_ou_set
                
                correspondances = pays_source.intersection(pays_reference)
                print(f"✅ {source}: {len(correspondances)}/{len(pays_source)} ({len(correspondances)/len(pays_source)*100:.1f}%)")


In [223]:

# =============================================================================
# UTILISATION DES FONCTIONS
# =============================================================================

if __name__ == "__main__":
    # 0.  Nettoyer le fichier de référence
    df_ref_propre = nettoyer_doublons_ref_pays()
    
    # 1. Standardiser les données brutes
    donnees_standardisees = appliquer_standardisation_complete(donnees_brutes)
    
    # 2. Appliquer le nettoyage sur les données standardisées
    donnees_propres = {}
    if donnees_standardisees:
        # Nettoyage religion
        donnees_propres['religion'] = nettoyer_donnees_religion(donnees_standardisees['religion'])
        
        # Nettoyage alimentaire (source prioritaire)
        donnees_propres['alimentaire'] = nettoyer_donnees_alimentaires(donnees_standardisees['alim_2017'])
        
        # Nettoyage population (source prioritaire) 
        donnees_propres['population'] = nettoyer_donnees_population(donnees_standardisees['population'])
        
        # Nettoyage World Bank
        donnees_propres['worldbank'] = transformer_donnees_worldbank(donnees_standardisees['worldbank'])
        
        # Nettoyage distances
        donnees_propres['distances'] = nettoyer_donnees_distances(donnees_standardisees['distances'])
        
        # Extraction religion récente
        donnees_propres['religion_recente'] = extraire_donnees_recentes_religion(donnees_propres['religion'])
        
    print("\n🎯 STANDARDISATION TERMINÉE - Prêt pour la fusion !")

🔍 NETTOYAGE DES DOUBLONS - FICHIER REF_PAYS
📊 Nombre total de lignes: 234

⚠️ DOUBLONS DÉTECTÉS sur 'country': 18 lignes
           country           pays
12       Australia      Australie
13       Australia      Australie
18           Benin          Bénin
19           Benin          Bénin
29         Bolivia        Bolivie
30         Bolivia        Bolivie
31          Brazil         Brésil
32          Brazil         Brésil
39          Canada         Canada
38          Canada         Canada
44   Côte d'Ivoire  Côte d'Ivoire
45   Côte d'Ivoire  Côte d'Ivoire
58         Germany      Allemagne
59         Germany      Allemagne
110      Kazakstan      Kazakstan
111      Kazakstan      Kazakstan
153        Nigeria        Nigéria
154        Nigeria        Nigéria

✅ NETTOYAGE TERMINÉ:
   Lignes avant: 234
   Lignes après: 225
   Doublons supprimés: 9
✅ Fichier nettoyé sauvegardé: ref_pays_clean.xlsx

🔧 STANDARDISATION COMPLÈTE AVEC RÉFÉRENCE

🗺️ CHARGEMENT FICHIER DE RÉFÉRENCE PAYS
----------

In [224]:
# DEBUG
def debug_pays_non_standardises(df, country_col, mapping_dict, source_name):
    """Debug les pays non trouvés dans le mapping"""
    print(f"\n🔍 DEBUG {source_name.upper()} - Pays non standardisés")
    print("-" * 50)
    
    # Identifier les pays non mappés
    pays_originaux = df[country_col].unique()
    pays_non_mappes = [pays for pays in pays_originaux if pays not in mapping_dict]
    
    print(f"• Pays non trouvés dans référence: {len(pays_non_mappes)}")
    
    if pays_non_mappes:
        print("• Liste des pays non mappés:")
        for pays in sorted(pays_non_mappes[:15]):  # 15 premiers
            nb_lignes = len(df[df[country_col] == pays])
            print(f"    {pays} ({nb_lignes} lignes)")
    
    return pays_non_mappes

# Rechargez votre mapping (du code précédent)
mapping_complet, df_ref = charger_reference_pays()

# DEBUG pour religion
if 'religion' in donnees_brutes:
    pays_religion_pb = debug_pays_non_standardises(
        donnees_brutes['religion'], 
        'Country', 
        mapping_complet, 
        'religion'
    )

# DEBUG pour alimentaire
if 'alim_2017' in donnees_brutes:
    pays_alim_pb = debug_pays_non_standardises(
        donnees_brutes['alim_2017'], 
        'Zone', 
        mapping_complet, 
        'alimentaire'
    )


🗺️ CHARGEMENT FICHIER DE RÉFÉRENCE PAYS
---------------------------------------------
✅ Fichier référence chargé: 225 pays
   Colonnes: ['iso2', 'iso3', 'cnum', 'country', 'pays', 'area', 'continent']
✅ Dictionnaire créé: 298 correspondances
   Français → Anglais: 225
   Mappings spéciaux: 78

🔍 DEBUG RELIGION - Pays non standardisés
--------------------------------------------------
• Pays non trouvés dans référence: 140
• Liste des pays non mappés:
    Albania (2 lignes)
    Algeria (2 lignes)
    All Asia-Pacific (2 lignes)
    All Europe (2 lignes)
    All Latin America-Caribbean (2 lignes)
    All Middle East-North Africa (2 lignes)
    All North America (2 lignes)
    All Sub-Saharan Africa (2 lignes)
    All World (2 lignes)
    Argentina (2 lignes)
    Armenia (2 lignes)
    Australia (2 lignes)
    Austria (2 lignes)
    Azerbaijan (2 lignes)
    Bahrain (2 lignes)

🔍 DEBUG ALIMENTAIRE - Pays non standardisés
--------------------------------------------------
• Pays non trouv

In [225]:


# ============================================================================
#  FONCTION DE DETAILS DES EXCLUSIONS DE DONNEES
# ============================================================================


def voir_details_exclusions(donnees_standardisees, donnees_brutes):
    """Analyse rapide des différences avant/après standardisation"""
    print("\n🔍 DÉTAILS DES EXCLUSIONS")
    print("=" * 40)
    
    config_colonnes = {
        'religion': 'Country', 'alim_2017': 'Zone', 'population': 'Zone', 
        'production': 'Zone', 'consommation': 'Zone', 'worldbank': 'Country Name',
        'distances': 'nom_pays'
    }
    
    for source in donnees_brutes:
        if source in config_colonnes and source in donnees_standardisees:
            col = config_colonnes[source]
            if col in donnees_brutes[source].columns:
                avant = set(donnees_brutes[source][col].dropna())
                apres = set(donnees_standardisees[source][col].dropna())
                exclus = avant - apres
                
                print(f"\n📍 {source}:")
                print(f"   Pays exclus ({len(exclus)}):")
                for pays in sorted(exclus):
                    nb = len(donnees_brutes[source][donnees_brutes[source][col] == pays])
                    print(f"      • {pays} ({nb} lignes)")

# Utilisation simple:
voir_details_exclusions(donnees_standardisees, donnees_brutes)


🔍 DÉTAILS DES EXCLUSIONS

📍 religion:
   Pays exclus (17):
      • All Asia-Pacific (2 lignes)
      • All Europe (2 lignes)
      • All Latin America-Caribbean (2 lignes)
      • All Middle East-North Africa (2 lignes)
      • All North America (2 lignes)
      • All Sub-Saharan Africa (2 lignes)
      • All World (2 lignes)
      • Bosnia-Herzegovina (2 lignes)
      • Brunei (2 lignes)
      • Channel Islands (2 lignes)
      • Czech Republic (2 lignes)
      • Democratic Republic of the Congo (2 lignes)
      • Eswatini (2 lignes)
      • Ivory Coast (2 lignes)
      • Kazakhstan (2 lignes)
      • Republic of the Congo (2 lignes)
      • United States (2 lignes)

📍 alim_2017:
   Pays exclus (123):
      • Afrique du Sud (1194 lignes)
      • Albanie (984 lignes)
      • Algérie (1061 lignes)
      • Allemagne (1123 lignes)
      • Antigua-et-Barbuda (1045 lignes)
      • Arabie saoudite (1035 lignes)
      • Argentine (1077 lignes)
      • Arménie (994 lignes)
      • Australie (

<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 6 - JOINTURES ET CRÉATION DU DATASET FINAL</h2>
</div>

In [226]:

# ============================================================================
#  FONCTION D'HARMONISATION DES PAYS
# ============================================================================

def creer_table_correspondance_pays(donnees):
    """
    Crée une table de correspondance pour harmoniser les noms de pays entre sources
    
    Args:
        donnees (dict): Dictionnaire des données nettoyées
        
    Returns:
        DataFrame: Table de correspondance des noms de pays
    """
    print("\n🗺️ CRÉATION TABLE DE CORRESPONDANCE DES PAYS")
    print("-" * 48)
    
    # Collecter tous les noms de pays uniques de chaque source
    pays_sources = {}
    
    # Alimentaire (source de référence principale)
    if 'alimentaire' in donnees and 'production' in donnees['alimentaire']:
        pays_sources['alimentaire'] = set(donnees['alimentaire']['production']['pays'].unique())
        print(f"• Alimentaire: {len(pays_sources['alimentaire'])} pays")
    
    # Population
    if 'population' in donnees:
        pays_sources['population'] = set(donnees['population']['pays'].unique()) 
        print(f"• Population: {len(pays_sources['population'])} pays")
    
    # Religion
    if 'religion_recente' in donnees:
        pays_sources['religion'] = set(donnees['religion_recente']['pays'].unique())
        print(f"• Religion: {len(pays_sources['religion'])} pays")
    
    
    # World Bank
    if 'worldbank' in donnees:
        pays_wb = set()
        for serie, df in donnees['worldbank'].items():
            pays_wb.update(df['pays'].unique())
        pays_sources['worldbank'] = pays_wb
        print(f"• World Bank: {len(pays_sources['worldbank'])} pays")
    
    # Distances
    if 'distances' in donnees:
        pays_sources['distances'] = set(donnees['distances']['pays_fr'].unique())
        print(f"• Distances: {len(pays_sources['distances'])} pays")
    
    # Créer la table de correspondance basée sur les sources alimentaires (prioritaires)
    if 'alimentaire' not in pays_sources:
        print("❌ Impossible de créer la correspondance sans données alimentaires")
        return None
    
    pays_reference = pays_sources['alimentaire']
    correspondance = []
    
    for pays_ref in sorted(pays_reference):
        correspondance.append({
            'pays_standard': pays_ref,
            'alimentaire': pays_ref,
            'population': pays_ref if pays_ref in pays_sources.get('population', set()) else '',
            'religion': pays_ref if pays_ref in pays_sources.get('religion', set()) else '',
            'worldbank': pays_ref if pays_ref in pays_sources.get('worldbank', set()) else '',
            'distances': pays_ref if pays_ref in pays_sources.get('distances', set()) else ''
        })
    
    df_correspondance = pd.DataFrame(correspondance)
    
    # Statistiques de correspondance
    print(f"\n📊 Statistiques de correspondance:")
    for source in ['population', 'religion', 'worldbank', 'distances']:
        if source in pays_sources:
            matches = (df_correspondance[source] != '').sum()
            total = len(df_correspondance)
            print(f"• {source}: {matches}/{total} ({matches/total*100:.1f}%)")
    
    return df_correspondance

In [227]:


# ============================================================================
#  FONCTION DE FUSION FINALE DES DONNEES
# ============================================================================
def fusionner_donnees_finales(donnees, correspondance):
   """
   Fusionne toutes les données en un dataset final avec suivi des pays perdus
   
   Args:
       donnees (dict): Données nettoyées
       correspondance (DataFrame): Table de correspondance des pays
       
   Returns:
       DataFrame: Dataset final fusionné
   """
   print("\n🔗 FUSION DES DONNÉES FINALES")
   print("-" * 32)
   
   # Commencer avec les données alimentaires (source prioritaire)
   if 'alimentaire' not in donnees:
       print("❌ Impossible de créer le dataset sans données alimentaires")
       return None
   
   # Base : données de production (tous les pays ont une production)
   df_final = donnees['alimentaire']['production'][['pays', 'production_tonnes', 'annee']].copy()
   pays_initiaux = set(df_final['pays'])
   print(f"• Base (production): {len(df_final)} pays")
   
   # Tracker pour les pays perdus
   pays_perdus_tracker = {'pays_initiaux': pays_initiaux}
   
   # Ajouter les autres données alimentaires
   for type_data in ['importations', 'exportations', 'consommation']:
       if type_data in donnees['alimentaire']:
           df_temp = donnees['alimentaire'][type_data][['pays', f'{type_data}_tonnes']]
           
           # Pays avant fusion
           pays_avant = set(df_final['pays'])
           
           df_final = df_final.merge(df_temp, on='pays', how='left')
           
           # Pays après fusion (ceux qui ont des données pour cette variable)
           pays_avec_donnees = set(df_final[df_final[f'{type_data}_tonnes'].notna()]['pays'])
           pays_perdus = pays_avant - pays_avec_donnees
           
           pays_perdus_tracker[f'perdus_{type_data}'] = pays_perdus
           
           print(f"• + {type_data}: {len(pays_avec_donnees)} pays avec données")
           if pays_perdus:
               print(f"   Pays sans données {type_data}: {len(pays_perdus)} ({', '.join(sorted(list(pays_perdus)[:3]))}{'...' if len(pays_perdus) > 3 else ''})")
   
   # Ajouter population (source prioritaire)
   if 'population' in donnees:
       df_pop = donnees['population'][['pays', 'population_milliers']]
       pays_avant = set(df_final['pays'])
       
       df_final = df_final.merge(df_pop, on='pays', how='left')
       
       pays_avec_donnees = set(df_final[df_final['population_milliers'].notna()]['pays'])
       pays_perdus = pays_avant - pays_avec_donnees
       pays_perdus_tracker['perdus_population'] = pays_perdus
       
       print(f"• + Population: {len(pays_avec_donnees)} pays avec données")
       if pays_perdus:
           print(f"   Pays sans données population: {len(pays_perdus)} ({', '.join(sorted(list(pays_perdus)[:3]))}{'...' if len(pays_perdus) > 3 else ''})")
   
   # Ajouter données World Bank
   if 'worldbank' in donnees:
       for serie, df_wb in donnees['worldbank'].items():
           df_temp = df_wb[['pays', serie]]
           pays_avant = set(df_final['pays'])
           
           df_final = df_final.merge(df_temp, on='pays', how='left')
           
           pays_avec_donnees = set(df_final[df_final[serie].notna()]['pays'])
           pays_perdus = pays_avant - pays_avec_donnees
           pays_perdus_tracker[f'perdus_{serie}'] = pays_perdus
           
           print(f"• + {serie}: {len(pays_avec_donnees)} pays avec données")
           if pays_perdus:
               print(f"   Pays sans données {serie}: {len(pays_perdus)} ({', '.join(sorted(list(pays_perdus)[:3]))}{'...' if len(pays_perdus) > 3 else ''})")
   
   # Ajouter données religieuses
   if 'religion_recente' in donnees:
       cols_religion = ['pays', 'religion_majoritaire', 'pct_religion_majoritaire', 'Pop_traitement_viande(%)']
       df_rel = donnees['religion_recente'][cols_religion]
       pays_avant = set(df_final['pays'])
       
       df_final = df_final.merge(df_rel, on='pays', how='left')
       
       pays_avec_donnees_religion = set(df_final[df_final['religion_majoritaire'].notna()]['pays'])
       pays_avec_donnees_traitement = set(df_final[df_final['Pop_traitement_viande(%)'].notna()]['pays'])
       pays_perdus = pays_avant - pays_avec_donnees_religion
       pays_perdus_tracker['perdus_religion'] = pays_perdus
       
       print(f"• + Religion: {len(pays_avec_donnees_religion)} pays avec données")
       print(f"• + Pop_traitement_viande(%): {len(pays_avec_donnees_traitement)} pays avec données")
       if pays_perdus:
           print(f"   Pays sans données religion: {len(pays_perdus)} ({', '.join(sorted(list(pays_perdus)[:3]))}{'...' if len(pays_perdus) > 3 else ''})")

   # Debug des valeurs religion et création variable booléenne
   if 'religion_majoritaire' in df_final.columns:
       print(f"\n🔍 Valeurs religion_majoritaire dans le dataset:")
       religions_counts = df_final['religion_majoritaire'].value_counts()
       for religion, count in religions_counts.items():
           print(f"   • {religion}: {count} pays")
   
   # Identifier les noms exacts des religions restrictives
   religions_restrictives = ['Musulmans', 'Juifs', 'Bouddhistes']
   
   # Pays avec restrictions
   pays_restrictions = df_final[df_final['religion_majoritaire'].isin(religions_restrictives)]['pays'].tolist()
   
   # Variable booléenne (True = pas de restrictions = bon pour export)
   df_final['traitement_particulier_viande'] = ~df_final['religion_majoritaire'].isin(religions_restrictives)

   # Conversion en texte explicite
   df_final['traitement_particulier_viande'] = df_final['traitement_particulier_viande'].map({
       True: 'Non',   # Pas de restrictions religieuses
       False: 'Oui'   # Restrictions religieuses présentes
   })
   
   if pays_restrictions:
       print(f"\n🚫 PAYS RESTRICTIONS RELIGIEUSES ({len(pays_restrictions)}):")
       for pays in sorted(pays_restrictions):
           religion = df_final[df_final['pays'] == pays]['religion_majoritaire'].iloc[0]
           print(f"   • {pays} ({religion})")
   
   print(f"• + Variable traitement_particulier_viande créée")
   # Sauvegarder la liste des pays exclus pour référence
   pays_exclus_religion = pays_restrictions.copy() if pays_restrictions else []
   print(f"\n📝 Liste sauvegardée: {len(pays_exclus_religion)} pays exclus pour restrictions religieuses")
   
   # Ajouter distances
   if 'distances' in donnees:
       # Utiliser le nom français comme clé de jointure principale
       df_dist = donnees['distances'][['pays_fr', 'distance_france_km', 'iso3_d', 'iso2', 'pays_en']].copy()
       df_dist.rename(columns={'pays_fr': 'pays'}, inplace=True)
       pays_avant = set(df_final['pays'])
       
       df_final = df_final.merge(df_dist, on='pays', how='left')
       
       pays_avec_donnees = set(df_final[df_final['distance_france_km'].notna()]['pays'])
       pays_perdus = pays_avant - pays_avec_donnees
       pays_perdus_tracker['perdus_distances'] = pays_perdus
       
       print(f"• + Distances: {len(pays_avec_donnees)} pays avec données")
       if pays_perdus:
           print(f"   Pays sans données distances: {len(pays_perdus)} ({', '.join(sorted(list(pays_perdus)[:3]))}{'...' if len(pays_perdus) > 3 else ''})")
   
   # Calculer des indicateurs dérivés
   print(f"\n📊 Calcul d'indicateurs dérivés:")
   
   # Delta import/export (exports - imports)
   if 'exportations_tonnes' in df_final.columns and 'importations_tonnes' in df_final.columns:
       df_final['delta_export_import'] = df_final['exportations_tonnes'] - df_final['importations_tonnes']
       print("• Delta export-import calculé")
   
   # Ratio consommation/production
   if 'consommation_tonnes' in df_final.columns and 'production_tonnes' in df_final.columns:
       df_final['ratio_consommation_production'] = df_final['consommation_tonnes'] / df_final['production_tonnes']
       print("• Ratio consommation/production calculé")
   
   # Population en millions pour lisibilité
   if 'population_milliers' in df_final.columns:
       df_final['population_millions'] = df_final['population_milliers'] / 1000
       print("• Population en millions ajoutée")
   
   # Détail des pays perdus par jointure
   print(f"\n📉 DÉTAIL DES PAYS PERDUS PAR JOINTURE:")
   for cle, pays_set in pays_perdus_tracker.items():
       if cle != 'pays_initiaux' and pays_set:
           variable = cle.replace('perdus_', '')
           print(f"\n• {variable.upper()} ({len(pays_set)} pays perdus):")
           for pays in sorted(pays_set):
               print(f"   - {pays}")

   # Calcul du total de pays conservés
   tous_pays_perdus = set()
   for k, v in pays_perdus_tracker.items():
       if k != 'pays_initiaux':
           tous_pays_perdus.update(v)

   print(f"\nPays conservés: {len(pays_initiaux) - len(tous_pays_perdus)} / {len(pays_initiaux)}")
   
   return df_final

In [228]:

# ============================================================================
#  FONCTION D'EVALUATION DE COMPLETUDE DES DONNEES
# ============================================================================

def evaluer_completude_donnees(df_final):
    """
    Évalue la complétude des données et identifie les pays avec datasets complets
    
    Args:
        df_final (DataFrame): Dataset final fusionné
        
    Returns:
        DataFrame: Dataset filtré sur les pays avec données suffisamment complètes
    """
    print("\n✅ ÉVALUATION DE LA COMPLÉTUDE DES DONNÉES")
    print("-" * 45)
    
    print(f"🔍 Dataset AVANT filtrage: {len(df_final)} pays")

    # Définir TOUTES les colonnes d'analyse (toutes au même niveau)
    colonnes_analyse = [
        'production_tonnes',
        'importations_tonnes', 
        'exportations_tonnes',
        'consommation_tonnes',
        'population_millions',
        'distance_france_km',
        'pib_total_usd',
        'croissance_pib_pct', 
        'urbanisation_pct',
        'stabilite_politique',
        'religion_majoritaire',
        'Pop_traitement_viande(%)' 
    ]
    
    print(f"📋 Variables d'analyse: {len(colonnes_analyse)}")
    for col in colonnes_analyse:
        if col in df_final.columns:
            nb_valeurs = df_final[col].notna().sum()
            print(f"• {col}: {nb_valeurs}/{len(df_final)} ({nb_valeurs/len(df_final)*100:.1f}%)")
    
    # Calculer score de complétude par pays (toutes variables égales)
    cols_existantes = [col for col in colonnes_analyse if col in df_final.columns]
    
    if cols_existantes:
        score_completude = df_final[cols_existantes].notna().sum(axis=1) / len(cols_existantes)
    else:
        score_completude = 0

    df_final['score_completude'] = score_completude
    
    # Filtrer sur les pays avec score = 100% (test ultra-qualité)
    masque_complet = (df_final['score_completude'] == 1.0)
    df_complet = df_final[masque_complet].copy()

    print(f"🔍 Dataset APRÈS filtrage: {len(df_complet)} pays")
    
    print(f"\n🎯 Résultats du filtrage:")
    print(f"• Pays avec score = 100%: {masque_complet.sum()}")
    print(f"• Pays dans dataset final: {len(df_complet)}")
    if len(df_complet) > 0:
        print(f"• Score moyen dataset final: {df_complet['score_completude'].mean():.2f}")
    
    return df_complet

# Application de la fusion
if donnees_propres:
    print("\n" + "="*70)
    print("🔗 PHASE DE FUSION DES DONNÉES") 
    print("="*70)
    
    # Créer table de correspondance
    correspondance_pays = creer_table_correspondance_pays(donnees_propres)
    
    # Fusionner les données
    dataset_final_brut = fusionner_donnees_finales(donnees_propres, correspondance_pays)
    
    if dataset_final_brut is not None:
        # Évaluer et filtrer sur complétude
        dataset_final = evaluer_completude_donnees(dataset_final_brut)




🔗 PHASE DE FUSION DES DONNÉES

🗺️ CRÉATION TABLE DE CORRESPONDANCE DES PAYS
------------------------------------------------
• Alimentaire: 168 pays
• Population: 238 pays
• Religion: 200 pays
• World Bank: 264 pays
• Distances: 224 pays

📊 Statistiques de correspondance:
• population: 168/168 (100.0%)
• religion: 158/168 (94.0%)
• worldbank: 152/168 (90.5%)
• distances: 159/168 (94.6%)

🔗 FUSION DES DONNÉES FINALES
--------------------------------
• Base (production): 168 pays
• + importations: 168 pays avec données
• + exportations: 135 pays avec données
   Pays sans données exportations: 33 (Guinea-Bissau, Haiti, Turkmenistan...)
• + consommation: 168 pays avec données
• + Population: 168 pays avec données
• + pib_total_usd: 152 pays avec données
   Pays sans données pib_total_usd: 16 (Egypt, Gambia, Turkey...)
• + croissance_pib_pct: 152 pays avec données
   Pays sans données croissance_pib_pct: 16 (Egypt, Gambia, Turkey...)
• + urbanisation_pct: 152 pays avec données
   Pays sans

In [229]:
# Autre fonction de debug
def analyser_correspondances_manquantes(donnees):
    """Debug pour identifier les noms de pays non correspondants"""
    
    # Pays alimentaire (référence)
    pays_alim = set(donnees['alimentaire']['production']['pays'].unique())
    
    # Pays religion
    if 'religion_recente' in donnees:
        pays_religion = set(donnees['religion_recente']['pays'].unique())
        manquants_religion = pays_alim - pays_religion
        print(f"\n🔍 PAYS ALIMENTAIRES ABSENTS DE RELIGION ({len(manquants_religion)}):")
        for pays in sorted(list(manquants_religion)[:10]):  # Premiers 10
            print(f"• {pays}")
    
    # Pays World Bank  
    if 'worldbank' in donnees:
        pays_wb = set()
        for serie, df in donnees['worldbank'].items():
            pays_wb.update(df['pays'].unique())
        manquants_wb = pays_alim - pays_wb
        print(f"\n🔍 PAYS ALIMENTAIRES ABSENTS DE WORLD BANK ({len(manquants_wb)}):")
        for pays in sorted(list(manquants_wb)[:10]):  # Premiers 10
            print(f"• {pays}")

# Analyse :
analyser_correspondances_manquantes(donnees_propres)


🔍 PAYS ALIMENTAIRES ABSENTS DE RELIGION (10):
• Antigua and Barbuda
• Czech Republic
• Dominica
• Korea
• Russian Federation
• République populaire démocratique de Corée
• Saint Kitts and Nevis
• Saint Lucia
• Saint Vincent and the Grenadines
• Viet Nam

🔍 PAYS ALIMENTAIRES ABSENTS DE WORLD BANK (16):
• Bahamas
• Egypt
• Gambia
• Korea
• République populaire démocratique de Corée
• Saint Lucia
• Taiwan
• Turkey
• Venezuela
• Yemen


<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 7 - VALIDATION ET EXPORT</h2>
</div>

In [230]:
def valider_dataset_final(df_final):
    """
    Valide le dataset final et génère les statistiques de validation
    
    Args:
        df_final (DataFrame): Dataset final
    """
    print("\n🔍 VALIDATION DU DATASET FINAL")
    print("-" * 35)
    
    print(f"📊 Dimensions finales: {df_final.shape[0]} pays × {df_final.shape[1]} variables")
    
    # Vérifier l'objectif 80-100 pays/
    nb_pays = len(df_final)
    if 80 <= nb_pays <= 100:
        print(f"✅ Objectif atteint: {nb_pays} pays (cible: 80-100)")
    elif nb_pays < 80:
        print(f"⚠️  Objectif partie2llement atteint: {nb_pays} pays (cible: 80-100)")
        print("💡 Considérer assouplir les critères de complétude si nécessaire")
    else:
        print(f"✅ Objectif dépassé: {nb_pays} pays (cible: 80-100)")
    
    
    # Répartition par religion majoritaire
    if 'religion_majoritaire' in df_final.columns:
        print(f"\n⛪ Répartition religieuse:")
        repartition_rel = df_final['religion_majoritaire'].value_counts()
        for religion, count in repartition_rel.items():
            print(f"• {religion}: {count} pays")
    
    # Statistiques économiques
        if 'pib_total_usd' in df_final.columns:
    # 🆕 Conversion numérique
            df_final['pib_total_usd'] = pd.to_numeric(df_final['pib_total_usd'], errors='coerce')
    
    df_pib = df_final.dropna(subset=['pib_total_usd'])
    if len(df_pib) > 0:
        print(f"\n💰 Économie (PIB total):")
        print(f"• Pays avec données PIB: {len(df_pib)}")
        print(f"• PIB médian: {df_pib['pib_total_usd'].median()/1e9:.1f} Mds USD")
        print(f"• PIB min: {df_pib['pib_total_usd'].min()/1e9:.1f} Mds USD")
        print(f"• PIB max: {df_pib['pib_total_usd'].max()/1e9:.1f} Mds USD")
    
    # Validation logique des données
    print(f"\n🔍 Validations logiques:")
    
    # Vérifier cohérence production/consommation
    if all(col in df_final.columns for col in ['production_tonnes', 'consommation_tonnes']):
        df_coherent = df_final.dropna(subset=['production_tonnes', 'consommation_tonnes'])
        ratios_extremes = ((df_coherent['consommation_tonnes'] / df_coherent['production_tonnes']) > 10) | \
                         ((df_coherent['consommation_tonnes'] / df_coherent['production_tonnes']) < 0.1)
        nb_extremes = ratios_extremes.sum()
        print(f"• Ratios consommation/production extrêmes: {nb_extremes} pays")
        if nb_extremes > 0:
            pays_extremes = df_coherent[ratios_extremes]['pays'].head(3).tolist()
            print(f"  Exemples: {', '.join(pays_extremes)}")
    
    # Vérifier distances cohérentes
    if 'distance_france_km' in df_final.columns:
        distances_valides = df_final['distance_france_km'].between(100, 20000)
        nb_valides = distances_valides.sum()
        print(f"• Distances cohérentes (100-20000 km): {nb_valides}/{len(df_final)}")

In [231]:


# ============================================================================
#  FONCTION DE GENERATION DE RESUME
# ============================================================================

def generer_resume_executif(df_final):
    """
    Génère un résumé exécutif pour les décideurs
    
    Args:
        df_final (DataFrame): Dataset final
    """
    print("\n" + "="*70)
    print("📋 RÉSUMÉ EXÉCUTIF - LA POULE QUI CHANTE")
    print("="*70)
    
    nb_pays = len(df_final)
    
    print(f"🎯 DATASET FINAL PRÊT POUR L'ANALYSE")
    print(f"• {nb_pays} pays avec données complètes")
    print(f"• {df_final.shape[1]} indicateurs par pays")
    print(f"• Données les plus récentes (2017-2018 principalement)")
    
    print(f"\n📊 INDICATEURS DISPONIBLES:")
    indicateurs_business = {
        'production_tonnes': 'Production locale (concurrence)',
        'importations_tonnes': 'Dépendance aux importations', 
        'exportations_tonnes': 'Capacité exportatrice',
        'consommation_tonnes': 'Taille du marché',
        'population_millions': 'Population totale',
        'pib_total_usd': 'Pouvoir économique',
        'distance_france_km': 'Coûts logistiques',
        'religion_majoritaire': 'Contraintes culturelles'
    }
    
    for col_tech, desc_business in indicateurs_business.items():
        if col_tech in df_final.columns:
            nb_donnees = df_final[col_tech].notna().sum()
            print(f"• {desc_business}: {nb_donnees}/{nb_pays} pays")
    
    # Insights préliminaires
    print(f"\n💡 INSIGHTS PRÉLIMINAIRES:")
    
    # Pays avec forte consommation et faible production (opportunités)
    if all(col in df_final.columns for col in ['production_tonnes', 'consommation_tonnes']):
        df_analyse = df_final.dropna(subset=['production_tonnes', 'consommation_tonnes'])
        opportunites = df_analyse[
            (df_analyse['consommation_tonnes'] > df_analyse['production_tonnes']) &
            (df_analyse['consommation_tonnes'] > df_analyse['consommation_tonnes'].quantile(0.7))
        ]
        print(f"• Marchés à fort potentiel (forte consommation, déficit production): {len(opportunites)} pays")
        if len(opportunites) > 0:
            top_opportunites = opportunites.nlargest(3, 'consommation_tonnes')['pays'].tolist()
            print(f"  Top 3: {', '.join(top_opportunites)}")
    
    # Pays proches géographiquement
    if 'distance_france_km' in df_final.columns:
        pays_proches = df_final[df_final['distance_france_km'] < 3000]
        print(f"• Marchés européens/proches (< 3000 km): {len(pays_proches)} pays")
    
    # Contraintes religieuses
    if 'religion_majoritaire' in df_final.columns:
        religions_compatibles = df_final[df_final['religion_majoritaire'].isin(['Chrétiens', 'Bouddhistes'])]
        print(f"• Marchés sans contraintes halal/casher: {len(religions_compatibles)} pays")
    
    print(f"\n🚀 PROCHAINES ÉTAPES:")
    print("1. Analyse en Composantes Principales (ACP)")
    print("2. Clustering des pays (CAH + K-means)")
    print("3. Identification des groupes de marchés cibles")
    print("4. Recommandations stratégiques par cluster")

In [232]:

# =================================================================================
#  FONCTION D'EXPORTATION DES DONNEES : COLONNES PERTINENTES SELECTIONNEES POUR ACP
# =================================================================================

def exporter_donnees(df_final, nom_fichier="dataset_final_poule_qui_chante.csv", epurer_pour_acp=True):
   """
   Exporte le dataset final et les métadonnées
   
   Args:
       df_final (DataFrame): Dataset final
       nom_fichier (str): Nom du fichier de sortie
       epurer_pour_acp (bool): Si True, supprime les colonnes inutiles pour l'ACP
   """
   print(f"\n💾 EXPORT DU DATASET FINAL")
   print("-" * 30)
   
   try:
       df_export = df_final.copy()  # Créer une copie pour ne pas modifier l'original
       
       if epurer_pour_acp:
           colonnes_a_supprimer = [
               'religion_majoritaire',         # → variable qualitative
               'pct_religion_majoritaire',     # → info religieuse déjà dans traitement_viande
               'traitement_particulier_viande', # → variable qualitative (Oui/Non)
               'zone_geographique',            # → laisse les clusters se dessiner
               'iso3_d', 'iso2',              # → codes techniques
               'population_milliers',          # → redondante avec population_millions
               'score_completude',             # → métrique interne
               'pays_en'                       # → redondant avec pays
           ]
           
           colonnes_existantes = [col for col in colonnes_a_supprimer if col in df_export.columns]
           df_export = df_export.drop(columns=colonnes_existantes)
           
           print(f"🧹 Dataset épuré: {len(colonnes_existantes)} colonnes supprimées")
           print(f"   Colonnes supprimées: {colonnes_existantes}")
       
       # Export dataset principal
       df_export.to_csv(nom_fichier, index=False, encoding='utf-8')
       print(f"✅ Dataset exporté: {nom_fichier}")
       print(f"• Dimensions: {df_export.shape}")
       print(f"• Taille fichier: ~{df_export.memory_usage(deep=True).sum() / 1024:.0f} KB")
       
       # Export métadonnées
       nom_meta = nom_fichier.replace('.csv', '_metadata.txt')
       with open(nom_meta, 'w', encoding='utf-8') as f:
           f.write("MÉTADONNÉES - DATASET FINAL LA POULE QUI CHANTE\n")
           f.write("=" * 50 + "\n\n")
           f.write(f"Date de création: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n")
           f.write(f"Nombre de pays: {len(df_export)}\n") 
           f.write(f"Nombre d'indicateurs: {df_export.shape[1]}\n")
           if epurer_pour_acp:
               f.write(f"Dataset épuré pour ACP: OUI\n")
               f.write(f"Colonnes supprimées: {len(colonnes_existantes)}\n")
           f.write("\n")
           
           f.write("DESCRIPTION DES COLONNES:\n")
           f.write("-" * 25 + "\n")
           descriptions = {
               'pays': 'Nom du pays',
               'production_tonnes': 'Production locale de volaille (tonnes)',
               'importations_tonnes': 'Importations de volaille (tonnes)', 
               'exportations_tonnes': 'Exportations de volaille (tonnes)',
               'consommation_tonnes': 'Consommation totale de volaille (tonnes)',
               'population_millions': 'Population totale (millions)',
               'pib_total_usd': 'PIB total (USD courants)',
               'croissance_pib_pct': 'Croissance PIB (%)',
               'urbanisation_pct': 'Population urbaine (%)',
               'stabilite_politique': 'Stabilité politique (percentile)',
               'distance_france_km': 'Distance depuis la France (km)',
               'delta_export_import': 'Balance commerciale volaille (exports-imports)',
               'ratio_consommation_production': 'Ratio consommation/production',
               'Pop_traitement_viande(%)': 'Pourcentage population avec restrictions religieuses viande (%)',
               'annee': 'Année de référence des données'
           }
           
           for col in df_export.columns:
               desc = descriptions.get(col, 'Variable calculée')
               f.write(f"{col}: {desc}\n")
       
       print(f"✅ Métadonnées exportées: {nom_meta}")
       
   except Exception as e:
       print(f"❌ Erreur export: {e}")

In [233]:
# ============================================================================
#  CALCUL DE LA COUVERTURE POPULATION MONDIALE
# ============================================================================

if 'dataset_final' in locals() and dataset_final is not None:
    print("\n" + "="*60)
    print("🌍 CALCUL DE LA COUVERTURE POPULATION MONDIALE")
    print("="*60)
    
    # Population totale du dataset final
    if 'population_millions' in dataset_final.columns:
        pop_dataset = dataset_final['population_millions'].sum()
        nb_pays_dataset = len(dataset_final)
        
        print(f"📊 Population dataset final:")
        print(f"• Nombre de pays: {nb_pays_dataset}")
        print(f"• Population totale: {pop_dataset:,.0f} millions")
        print(f"• Population totale: {pop_dataset*1000:,.0f} milliers")
        
        # Population mondiale de référence (2020-2022)
        population_mondiale_2022 = 7975  # millions (source: Banque Mondiale)
        
        # Calcul du ratio de couverture
        ratio_couverture = (pop_dataset / population_mondiale_2022) * 100
        
        print(f"\n🌎 Couverture mondiale:")
        print(f"• Population mondiale 2022: {population_mondiale_2022:,.0f} millions")
        print(f"• Couverture de l'étude: {ratio_couverture:.1f}%")
        
        # Analyse qualitative
        if ratio_couverture >= 70:
            print(f"✅ Excellente représentativité mondiale ({ratio_couverture:.1f}%)")
        elif ratio_couverture >= 50:
            print(f"✅ Bonne représentativité mondiale ({ratio_couverture:.1f}%)")
        elif ratio_couverture >= 30:
            print(f"⚠️  Représentativité correcte ({ratio_couverture:.1f}%)")
        else:
            print(f"❌ Représentativité limitée ({ratio_couverture:.1f}%)")
        
        # Top 5 des pays les plus peuplés dans l'étude
        if len(dataset_final) >= 5:
            top_pays = dataset_final.nlargest(5, 'population_millions')[['pays', 'population_millions']]
            print(f"\n🏆 Top 5 des pays les plus peuplés dans l'étude:")
            for idx, (_, row) in enumerate(top_pays.iterrows(), 1):
                print(f"   {idx}. {row['pays']}: {row['population_millions']:,.0f} millions")
        
        # Stocker les résultats pour usage ultérieur
        stats_population = {
            'population_dataset_millions': pop_dataset,
            'nb_pays_dataset': nb_pays_dataset,
            'population_mondiale_millions': population_mondiale_2022,
            'ratio_couverture_pct': ratio_couverture
        }
        
        print(f"\n💾 Statistiques sauvegardées pour référence")
        
    else:
        print("❌ Colonne 'population_millions' non trouvée dans le dataset")
        stats_population = None

else:
    print("❌ Dataset final non disponible pour le calcul de couverture")
    stats_population = None


🌍 CALCUL DE LA COUVERTURE POPULATION MONDIALE
📊 Population dataset final:
• Nombre de pays: 111
• Population totale: 6,000 millions
• Population totale: 6,000,171 milliers

🌎 Couverture mondiale:
• Population mondiale 2022: 7,975 millions
• Couverture de l'étude: 75.2%
✅ Excellente représentativité mondiale (75.2%)

🏆 Top 5 des pays les plus peuplés dans l'étude:
   1. China: 1,428 millions
   2. India: 1,353 millions
   3. United States of America: 327 millions
   4. Indonesia: 268 millions
   5. Pakistan: 212 millions

💾 Statistiques sauvegardées pour référence


In [234]:

# Supprimer la colonne annee du dataset_final si elle existe
if 'annee' in dataset_final.columns:
    dataset_final = dataset_final.drop('annee', axis=1)
    print(f"Colonne 'annee' supprimée. Nouvelles dimensions: {dataset_final.shape}")
else:
    print("Colonne 'annee' non trouvée dans le dataset")

# ============================================================================
#  APPLICATTION DE LA VALIDATION ET EXPORT
# ============================================================================


if 'dataset_final' in locals() and dataset_final is not None:
    print("\n" + "="*70)
    print("✅ PHASE DE VALIDATION ET EXPORT")
    print("="*70)
    
    # Validation
    valider_dataset_final(dataset_final)
    
    # Résumé exécutif
    generer_resume_executif(dataset_final)
    
    # Export
    exporter_donnees(dataset_final)

# ============================================================================
# 8. CONCLUSION ET PROCHAINES ÉTAPES
# ============================================================================

print("\n" + "="*70)
print("🎉 TRAITEMENT DES DONNÉES TERMINÉ")
print("="*70)

if 'dataset_final' in locals():
    print(f"✅ Dataset final créé avec succès")
    print(f"• {len(dataset_final)} pays analysables")
    print(f"• {dataset_final.shape[1]} indicateurs par pays")     
    print(f"• Données exportées et prêtes pour l'ACP/Clustering") # ← Fichier exporté (épuré)
    
    print(f"\n📋 FICHIERS GÉNÉRÉS:")
    print("• dataset_final_poule_qui_chante.csv - Dataset principal")
    print("• dataset_final_poule_qui_chante_metadata.txt - Documentation")
    
    print(f"\n🔮 PROCHAINES ÉTAPES RECOMMANDÉES:")
    print("1. Créer un nouveau notebook pour l'ACP")
    print("2. Réaliser la réduction des dimensions") 
    print("3. Appliquer le clustering (CAH + K-means)")
    print("4. Interpréter les clusters et formuler recommandations")
    
    print(f"\n💡 POUR RÉUTILISATION FUTURE:")
    print("• Modifier les chemins des fichiers CSV dans la fonction charger_donnees()")
    print("• Ajuster les critères de complétude si besoin (fonction evaluer_completude_donnees)")
    print("• Personnaliser les indicateurs dérivés selon nouveaux besoins")
        
else:
    print("❌ Erreur dans le traitement - Vérifiez les données d'entrée")

print("\n" + "="*70)

Colonne 'annee' supprimée. Nouvelles dimensions: (111, 22)

✅ PHASE DE VALIDATION ET EXPORT

🔍 VALIDATION DU DATASET FINAL
-----------------------------------
📊 Dimensions finales: 111 pays × 22 variables
✅ Objectif dépassé: 111 pays (cible: 80-100)

⛪ Répartition religieuse:
• Chrétiens: 77 pays
• Musulmans: 23 pays
• Non affilié: 6 pays
• Hindous: 2 pays
• Bouddhistes: 2 pays
• Juifs: 1 pays

💰 Économie (PIB total):
• Pays avec données PIB: 111
• PIB médian: 71.1 Mds USD
• PIB min: 0.9 Mds USD
• PIB max: 20656.5 Mds USD

🔍 Validations logiques:
• Ratios consommation/production extrêmes: 6 pays
  Exemples: Hong Kong, Gabon, Solomon Islands
• Distances cohérentes (100-20000 km): 111/111

📋 RÉSUMÉ EXÉCUTIF - LA POULE QUI CHANTE
🎯 DATASET FINAL PRÊT POUR L'ANALYSE
• 111 pays avec données complètes
• 22 indicateurs par pays
• Données les plus récentes (2017-2018 principalement)

📊 INDICATEURS DISPONIBLES:
• Production locale (concurrence): 111/111 pays
• Dépendance aux importations: 111/1

In [235]:
dataset_final

,pays,production_tonnes,importations_tonnes,exportations_tonnes,consommation_tonnes,population_milliers,pib_total_usd,croissance_pib_pct,urbanisation_pct,stabilite_politique,...,Pop_traitement_viande(%),traitement_particulier_viande,distance_france_km,iso3_d,iso2,pays_en,delta_export_import,ratio_consommation_production,population_millions,score_completude
1,South Africa,1667.0,514.0,63.0,2035.0,57792.518,4.052607e+11,1.556784,66.355,-0.237547,...,1.705591,Non,9353.5520,ZAF,ZA,South Africa,-451.0,1.220756,57.792518,1.0
2,Albania,13.0,38.0,0.0,47.0,2882.740,1.537951e+10,3.671419,60.319,0.366779,...,74.517433,Oui,1603.5340,ALB,AL,Albania,-38.0,3.615385,2.882740,1.0
3,Algeria,275.0,2.0,0.0,264.0,42228.408,1.945545e+11,1.400000,72.629,-0.842108,...,98.397478,Oui,1340.3900,DZA,DZ,Algeria,-2.0,0.960000,42.228408,1.0
4,Germany,1514.0,842.0,646.0,1609.0,83124.418,4.052008e+12,1.116443,77.312,0.577865,...,7.034101,Non,439.8984,DEU,DE,Germany,-196.0,1.062748,83.124418,1.0
5,Angola,42.0,277.0,0.0,315.0,30809.787,7.945069e+10,-1.316362,65.514,-0.347209,...,0.259246,Non,6510.3220,AGO,AO,Angola,-277.0,7.500000,30.809787,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,Trinidad and Tobago,61.0,23.0,0.0,75.0,1389.843,2.388891e+10,-0.985525,53.184,0.230739,...,5.513437,Non,7225.3250,TTO,TT,Trinidad and Tobago,-23.0,1.229508,1.389843,1.0
157,Tunisia,213.0,0.0,4.0,195.0,11565.201,4.268650e+10,2.624930,68.945,-0.868207,...,99.312017,Oui,1483.8280,TUN,TN,Tunisia,4.0,0.915493,11.565201,1.0
160,Ukraine,1211.0,121.0,273.0,1006.0,44246.156,1.308911e+11,3.488362,69.352,-1.872179,...,0.997956,Non,2025.1440,UKR,UA,Ukraine,152.0,0.830718,44.246156,1.0
161,Uruguay,33.0,3.0,3.0,31.0,3449.285,6.534458e+10,0.164778,95.334,1.024146,...,0.567564,Non,10972.4200,URY,UY,Uruguay,0.0,0.939394,3.449285,1.0


<div style="background-color: RGB(0,150,250);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 7.5 - VERIFICATION FINALE</h2>
</div>

In [236]:

# ============================================================================
#  FONCTION D'ANALYSE DES EXCLUSIONS
# ============================================================================

def analyser_exclusions_complete(dataset_final):
    """
    Compare le dataset final avec le fichier de référence
    """
    print("🔍 ANALYSE COMPLÈTE DES EXCLUSIONS")
    print("=" * 50)
    
    # 1. Charger le fichier de référence nettoyé
    df_ref = pd.read_excel('ref_pays_clean.xlsx')
    pays_reference = set(df_ref['country'].dropna().str.strip())
    print(f"📊 Pays dans fichier référence: {len(pays_reference)}")
    
    # 2. Pays dans dataset final
    pays_finaux = set(dataset_final['pays'].unique())
    print(f"📊 Pays dans dataset final: {len(pays_finaux)}")
    
    # 3. Pays exclus = Référence - Dataset final
    pays_exclus = pays_reference - pays_finaux
    print(f"📊 Pays exclus au total: {len(pays_exclus)}")
    
    # 4. Liste complète
    print(f"\n🚫 LISTE COMPLÈTE DES PAYS EXCLUS:")
    for pays in sorted(pays_exclus):
        print(f"   • {pays}")
        
    return pays_exclus, pays_reference, pays_finaux

# Appel avec le dataset en paramètre
exclusions_data = analyser_exclusions_complete(dataset_final)

🔍 ANALYSE COMPLÈTE DES EXCLUSIONS
📊 Pays dans fichier référence: 225
📊 Pays dans dataset final: 111
📊 Pays exclus au total: 115

🚫 LISTE COMPLÈTE DES PAYS EXCLUS:
   • Afghanistan
   • Andorra
   • Anguilla
   • Antigua and Barbuda
   • Aruba
   • Bahamas
   • Bahrain
   • Bangladesh
   • Bermuda
   • Bhutan
   • British Virgin Islands
   • Brunei Darussalam
   • Burkina Faso
   • Burma
   • Burundi
   • Cambodia
   • Cape Verde
   • Cayman Islands
   • Central African Republic
   • Christmas Island
   • Cocos (Keeling) Islands
   • Comoros
   • Congo
   • Congo (Democratic Republic of the)
   • Cook Islands
   • Cuba
   • Czech Republic
   • Côte d'Ivoire
   • Djibouti
   • Dominica
   • East Timor
   • Egypt
   • Equatorial Guinea
   • Eritrea
   • Ethiopia
   • Falkland Islands
   • Faroe Islands
   • French Guiana
   • French Polynesia
   • French Southern Antartic territories
   • Gambia
   • Gibraltar
   • Greenland
   • Grenada
   • Guadeloupe
   • Guinea
   • Guinea-Bissau
   •

In [237]:

# ============================================================================
#  EXEMPLE DE FONCTION D'ANALYSE CAS PAR CAS
# ============================================================================

def identifier_source_belgium_luxembourg():
    """
    Cherche dans toutes les sources de données d'où vient 'Belgium and Luxembourg'
    """
    print("🔍 RECHERCHE DE 'Belgium and Luxembourg' DANS TOUTES LES SOURCES")
    print("=" * 60)
    
    # Liste des fichiers à vérifier
    fichiers_a_verifier = [
        'Religious_Composition_2010-2020_percentages.csv',
        'DisponibiliteAlimentaire_2017.csv', 
        'Population_2000_2018.csv',
        'FAOSTAT_data_production_poulet.csv',
        'FAOSTAT_data_consommation_poulet_2010_2018.csv',
        'FAOSTAT_data_import_export_poulet.csv',
        'WORLDBANK_data_country_2010_2018.csv'
    ]
    
    for fichier in fichiers_a_verifier:
        try:
            print(f"\n📁 Vérification: {fichier}")
            df = pd.read_csv(fichier)
            
            # Chercher toutes les colonnes qui pourraient contenir des noms de pays
            colonnes_pays = [col for col in df.columns if any(mot in col.lower() for mot in ['country', 'zone', 'pays', 'name'])]
            
            for col in colonnes_pays:
                valeurs_uniques = df[col].dropna().astype(str)
                belgium_matches = valeurs_uniques[valeurs_uniques.str.contains('Belgium', case=False, na=False)]
                
                if len(belgium_matches) > 0:
                    print(f"   ✅ Trouvé dans colonne '{col}':")
                    for match in belgium_matches.unique():
                        print(f"      • {match}")
                        
        except Exception as e:
            print(f"   ❌ Erreur lecture {fichier}: {e}")
    
    # Vérifier aussi le fichier Excel des distances
    try:
        print(f"\n📁 Vérification: Distance_france_nom_iso.xlsx")
        df_dist = pd.read_excel('Distance_france_nom_iso.xlsx', sheet_name='dist_cepii')
        colonnes_pays = ['nom_pays', 'country_name']
        
        for col in colonnes_pays:
            if col in df_dist.columns:
                belgium_matches = df_dist[col].dropna().astype(str)
                belgium_matches = belgium_matches[belgium_matches.str.contains('Belgium', case=False, na=False)]
                
                if len(belgium_matches) > 0:
                    print(f"   ✅ Trouvé dans colonne '{col}':")
                    for match in belgium_matches.unique():
                        print(f"      • {match}")
                        
    except Exception as e:
        print(f"   ❌ Erreur lecture fichier distances: {e}")

# Lancer la recherche
identifier_source_belgium_luxembourg()

🔍 RECHERCHE DE 'Belgium and Luxembourg' DANS TOUTES LES SOURCES

📁 Vérification: Religious_Composition_2010-2020_percentages.csv
   ✅ Trouvé dans colonne 'Country':
      • Belgium

📁 Vérification: DisponibiliteAlimentaire_2017.csv

📁 Vérification: Population_2000_2018.csv

📁 Vérification: FAOSTAT_data_production_poulet.csv

📁 Vérification: FAOSTAT_data_consommation_poulet_2010_2018.csv

📁 Vérification: FAOSTAT_data_import_export_poulet.csv

📁 Vérification: WORLDBANK_data_country_2010_2018.csv
   ✅ Trouvé dans colonne 'Country Name':
      • Belgium

📁 Vérification: Distance_france_nom_iso.xlsx
   ✅ Trouvé dans colonne 'country_name':
      • Belgium


In [238]:

# ============================================================================
#  FONCTION D'IDENTIFICATION DES SOURCES DES PAYS MANQUANTS
# ============================================================================

def identifier_source_pays_manques():
    """
    Trouve dans quelles sources apparaissent ces pays importants
    """
    pays_recherches = ['Korea', 'Korea, Dem. People\'s Rep. of', 'Turkey', 'United States of America', 'Russian Federation']
    
    for pays in pays_recherches:
        print(f"\n🔍 Recherche: '{pays}'")
        
        # Chercher dans toutes vos sources
        for source_key, df in donnees_brutes.items():
            # Identifier la colonne pays selon la source
            col_pays = {'religion': 'Country', 'alim_2017': 'Zone', 'worldbank': 'Country Name'}.get(source_key)
            
            if col_pays and col_pays in df.columns:
                if pays in df[col_pays].values:
                    print(f"   ✅ Trouvé dans: {source_key}")

# Lancer la recherche
identifier_source_pays_manques()


🔍 Recherche: 'Korea'

🔍 Recherche: 'Korea, Dem. People's Rep. of'

🔍 Recherche: 'Turkey'
   ✅ Trouvé dans: religion

🔍 Recherche: 'United States of America'

🔍 Recherche: 'Russian Federation'
   ✅ Trouvé dans: worldbank
